## Cell 2: Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import collections
from evaluate import load
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast, get_linear_schedule_with_warmup
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments, AutoTokenizer
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np

2025-10-15 17:43:19.114029: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760528599.131502   21500 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760528599.136731   21500 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760528599.149735   21500 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760528599.149752   21500 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760528599.149754   21500 computation_placer.cc:177] computation placer alr

## Cell 3: Configuration for "TinyBERT"

In [2]:
class TinyBertConfig:
    vocab_size = 30522
    hidden_size = 768
    num_hidden_layers = 12
    num_attention_heads = 12
    intermediate_size = 3072
    max_position_embeddings = 128
    type_vocab_size = 2
    hidden_dropout_prob = 0.1
    attention_probs_dropout_prob = 0.1
    initializer_range = 0.02

config = TinyBertConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Cell 4: Model Architecture - Building Blocks

In [3]:
class BertEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=0)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.register_buffer("position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)))

    def forward(self, input_ids, token_type_ids=None):
        seq_length = input_ids.size(1)
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        
        word_embeds = self.word_embeddings(input_ids)
        position_ids = self.position_ids[:, :seq_length]
        position_embeds = self.position_embeddings(position_ids)
        token_type_embeds = self.token_type_embeddings(token_type_ids)
        
        embeddings = word_embeds + position_embeds + token_type_embeds
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)
        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_scores = attention_scores + attention_mask

        attention_probs = nn.Softmax(dim=-1)(attention_scores)
        attention_probs = self.dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        return context_layer

class BertEncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadSelfAttention(config)
        self.attention_dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.attention_dropout = nn.Dropout(config.hidden_dropout_prob)
        self.attention_layernorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        
        self.ffn = nn.Linear(config.hidden_size, config.intermediate_size)
        self.ffn_output = nn.Linear(config.intermediate_size, config.hidden_size)
        self.ffn_dropout = nn.Dropout(config.hidden_dropout_prob)
        self.ffn_layernorm = nn.LayerNorm(config.hidden_size, eps=1e-12)

    def forward(self, hidden_states, attention_mask):
        attention_output = self.attention(hidden_states, attention_mask)
        attention_output = self.attention_dense(attention_output)
        attention_output = self.attention_dropout(attention_output)
        attention_output = self.attention_layernorm(hidden_states + attention_output)
        
        ffn_hidden = F.gelu(self.ffn(attention_output))
        ffn_output = self.ffn_output(ffn_hidden)
        ffn_output = self.ffn_dropout(ffn_output)
        encoder_output = self.ffn_layernorm(attention_output + ffn_output)
        return encoder_output

## Cell 5: Model Architecture - Main TinyBERT Model

In [4]:
class TinyBERT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embeddings = BertEmbeddings(config)
        self.encoder_blocks = nn.ModuleList([BertEncoderBlock(config) for _ in range(config.num_hidden_layers)])

    def forward(self, input_ids, token_type_ids=None, attention_mask=None):
        if attention_mask is None:
            attention_mask = (input_ids != 0).float()
        
        extended_attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0
        
        embedding_output = self.embeddings(input_ids, token_type_ids)
        
        hidden_states = embedding_output
        for i, layer_module in enumerate(self.encoder_blocks):
            hidden_states = layer_module(hidden_states, extended_attention_mask)
        
        return hidden_states

class MLMHead(nn.Module):
    def __init__(self, config, word_embedding_weight):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.decoder = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.decoder.weight = word_embedding_weight
        self.bias = nn.Parameter(torch.zeros(config.vocab_size))
        self.decoder.bias = self.bias

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = F.gelu(hidden_states)
        hidden_states = self.layer_norm(hidden_states)
        prediction_scores = self.decoder(hidden_states)
        return prediction_scores
        
class MaskedLanguageModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = TinyBERT(config)
        self.cls = MLMHead(config, self.bert.embeddings.word_embeddings.weight)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        sequence_output = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        prediction_scores = self.cls(sequence_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(prediction_scores.view(-1, config.vocab_size), labels.view(-1))
            
        return prediction_scores, loss

## Cell 6: Data Preparation for Pre-training

In [7]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=config.max_position_embeddings, padding='max_length')

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_dataset.set_format(type='torch')

def data_collator_for_mlm(features):
    batch = {key: torch.stack([f[key] for f in features]) for key in features[0].keys()}
    batch['labels'] = batch['input_ids'].clone()
    
    probability_matrix = torch.full(batch['labels'].shape, 0.15)
    special_tokens_mask = [
        val in [tokenizer.sep_token_id, tokenizer.cls_token_id, tokenizer.pad_token_id]
        for val in batch['labels'].view(-1)
    ]
    special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
    special_tokens_mask = special_tokens_mask.view(batch['labels'].shape)
    probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

    masked_indices = torch.bernoulli(probability_matrix).bool()
    batch['labels'][~masked_indices] = -100

    indices_replaced = torch.bernoulli(torch.full(batch['labels'].shape, 0.8)).bool() & masked_indices
    batch['input_ids'][indices_replaced] = tokenizer.mask_token_id

    indices_random = torch.bernoulli(torch.full(batch['labels'].shape, 0.5)).bool() & masked_indices & ~indices_replaced
    random_words = torch.randint(len(tokenizer), batch['labels'].shape, dtype=torch.long)
    batch['input_ids'][indices_random] = random_words[indices_random]
    
    return batch

/home/cseru/.pyenv/versions/3.10.12/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Cell 7: Pre-training on the "wikitext-2" dataset

In [6]:
CHECKPOINT_PATH = "tinybert_pretrained_intermediate.pth"
start_epoch = 0

mlm_model = MaskedLanguageModel(config).to(device)
optimizer = AdamW(mlm_model.parameters(), lr=5e-5)
scaler = torch.cuda.amp.GradScaler()

if os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint found at {CHECKPOINT_PATH}. Resuming training.")
    checkpoint = torch.load(CHECKPOINT_PATH)
    
    mlm_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    
    print(f"Resuming from epoch {start_epoch + 1}")
else:
    print("No checkpoint found. Starting training from scratch.")

train_dataloader = DataLoader(tokenized_dataset, batch_size=32, shuffle=True, collate_fn=data_collator_for_mlm, num_workers=2)

num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
num_warmup_steps = 0

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

if start_epoch > 0:
    steps_to_advance = start_epoch * len(train_dataloader)
    for _ in range(steps_to_advance):
        scheduler.step()

mlm_model.train()
progress_bar = tqdm(range(num_training_steps), initial=start_epoch * len(train_dataloader))

for epoch in range(start_epoch, num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        with torch.cuda.amp.autocast():
            _, loss = mlm_model(**batch)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        
        progress_bar.update(1)
        progress_bar.set_description(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

    print(f"\nEpoch {epoch+1} finished. Saving intermediate checkpoint...")
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': mlm_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'loss': loss.item(),
    }
    
    torch.save(checkpoint, CHECKPOINT_PATH)
    print(f"Intermediate checkpoint saved to {CHECKPOINT_PATH}")

print("\nTraining complete. Saving final model weights.")
torch.save(mlm_model.bert.state_dict(), "tinybert_pretrained.pth")
print("Final model weights saved to tinybert_pretrained.pth")

Checkpoint found at tinybert_pretrained_intermediate.pth. Resuming training.
Resuming from epoch 3


/home/cseru/.pyenv/versions/3.10.12/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:139: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


 40%|####      | 2296/5740 [00:00<?, ?it/s]


Epoch 3 finished. Saving intermediate checkpoint...
Intermediate checkpoint saved to tinybert_pretrained_intermediate.pth

Epoch 4 finished. Saving intermediate checkpoint...
Intermediate checkpoint saved to tinybert_pretrained_intermediate.pth

Epoch 5 finished. Saving intermediate checkpoint...
Intermediate checkpoint saved to tinybert_pretrained_intermediate.pth

Training complete. Saving final model weights.
Final model weights saved to tinybert_pretrained.pth


## Qualitative Evaluation (Fill-Mask Examples)

In [ ]:
config = TinyBertConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model = TinyBERT(config)

model_weights_path = "tinybert_pretrained.pth"
bert_model.load_state_dict(torch.load(model_weights_path))

mlm_model_eval = MaskedLanguageModel(config)
mlm_model_eval.bert = bert_model
mlm_model_eval.to(device)
mlm_model_eval.eval()

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

sentences = [
    "The capital of France is [MASK].",
    "He went to the library to read a [MASK].",
    "The cat chased the [MASK] up the tree.",
    "Water boils at 100 degrees [MASK].",
    "The sun rises in the [MASK] and sets in the west.",
    "A doctor's main job is to [MASK] patients.",
    "The opposite of hot is [MASK].",
    "To solve this problem, you need to [MASK] hard.",
    "An apple a day keeps the [MASK] away.",
    "The planets in our solar system orbit the [MASK]."
]

for sentence in sentences:
    print(f"Input: {sentence}")
    
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    input_ids = inputs['input_ids']
    
    with torch.no_grad():
        logits, _ = mlm_model_eval(**inputs)

    mask_token_index = torch.where(input_ids == tokenizer.mask_token_id)[1]
    mask_logits = logits[0, mask_token_index, :]
    
    top_5_tokens = torch.topk(mask_logits, 5, dim=1).indices[0].tolist()
    
    predictions = []
    for token in top_5_tokens:
        predictions.append(tokenizer.decode([token]))
        
    print(f"Top 5 Predictions: {predictions}\n")

Input: The capital of France is [MASK].
Top 5 Predictions: ['ramon', 'motorsports', 'stated', '[unused588]', 'motioned']

Input: He went to the library to read a [MASK].
Top 5 Predictions: ['ramon', 'motorsports', 'stated', '[unused588]', 'motioned']

Input: The cat chased the [MASK] up the tree.
Top 5 Predictions: ['ramon', 'motorsports', '[unused588]', 'stated', 'scorer']

Input: Water boils at 100 degrees [MASK].
Top 5 Predictions: ['ramon', 'motorsports', 'stated', '[unused588]', 'scorer']

Input: The sun rises in the [MASK] and sets in the west.
Top 5 Predictions: ['ramon', 'motorsports', 'stated', '[unused588]', 'scorer']

Input: A doctor's main job is to [MASK] patients.
Top 5 Predictions: ['ramon', 'motorsports', 'motioned', 'stated', '[unused588]']

Input: The opposite of hot is [MASK].
Top 5 Predictions: ['ramon', 'motorsports', 'stated', '[unused588]', 'motioned']

Input: To solve this problem, you need to [MASK] hard.
Top 5 Predictions: ['ramon', 'motorsports', 'stated', '[

## Cell 8: Fine-Tuning Task 1: Sentiment Classification (IMDB)

In [34]:
class BertForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels):
        super().__init__()
        self.bert = TinyBERT(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, label=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs[:, 0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if label is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, label)
            
        return logits, loss

In [12]:
imdb_dataset = load_dataset('imdb')
train_dataset = imdb_dataset['train'].shuffle(seed=42).select(range(5000))
test_dataset = imdb_dataset['test'].shuffle(seed=42).select(range(2000))

def tokenize_imdb(examples):
    return tokenizer(examples['text'], truncation=True, max_length=config.max_position_embeddings, padding='max_length')

train_dataset = train_dataset.map(tokenize_imdb, batched=True, remove_columns=['text'])
test_dataset = test_dataset.map(tokenize_imdb, batched=True, remove_columns=['text'])
train_dataset.set_format('torch')
test_dataset.set_format('torch')

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32)
test_dataloader = DataLoader(test_dataset, batch_size=32)

sentiment_model = BertForSequenceClassification(config, num_labels=2).to(device)
sentiment_model.bert.load_state_dict(torch.load("tinybert_pretrained.pth"))

optimizer = AdamW(sentiment_model.parameters(), lr=2e-5)
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
scaler = torch.cuda.amp.GradScaler()

print("Starting Sentiment Classification Fine-tuning...")
sentiment_model.train()
for epoch in range(num_epochs):
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            _, loss = sentiment_model(**batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

sentiment_model.eval()
total_correct = 0
total_samples = 0
for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        logits, _ = sentiment_model(**batch)
    predictions = torch.argmax(logits, dim=-1)
    total_correct += (predictions == batch['label']).sum().item()
    total_samples += len(batch['label'])

tinybert_sentiment_accuracy = total_correct / total_samples
print(f"TinyBERT Sentiment Classification Accuracy: {tinybert_sentiment_accuracy:.4f}")
torch.save(sentiment_model.state_dict(), "tinybert_sentiment.pth")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Starting Sentiment Classification Fine-tuning...


Epoch 1:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/157 [00:00<?, ?it/s]

TinyBERT Sentiment Classification Accuracy: 0.6595


## Sentiment Analysis - Benchmark

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_metric

model_checkpoint = "distilbert-base-uncased"
metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
training_args = TrainingArguments(
    output_dir="./results_sentiment",
    evaluation_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting Benchmark Sentiment Classification...")
trainer.train()
distilbert_sentiment_results = trainer.evaluate()

/tmp/ipykernel_4116/393230262.py:5: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("accuracy")
/home/cseru/.pyenv/versions/3.10.12/lib/python3.10/site-packages/datasets/load.py:759: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/home/cseru/.pyenv/versions/3.10.12/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in ve


Starting Benchmark Sentiment Classification...


  0%|          | 0/785 [00:00<?, ?it/s]

{'loss': 0.5177, 'learning_rate': 4.687898089171975e-05, 'epoch': 0.32}
{'loss': 0.4258, 'learning_rate': 4.375796178343949e-05, 'epoch': 0.64}
{'loss': 0.392, 'learning_rate': 4.057324840764332e-05, 'epoch': 0.96}


  0%|          | 0/63 [00:00<?, ?it/s]

...

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.6057778596878052, 'eval_accuracy': 0.8495, 'eval_runtime': 7.3077, 'eval_samples_per_second': 273.684, 'eval_steps_per_second': 8.621, 'epoch': 4.0}
{'loss': 0.0384, 'learning_rate': 8.726114649681529e-06, 'epoch': 4.14}
{'loss': 0.0185, 'learning_rate': 5.541401273885351e-06, 'epoch': 4.46}
{'loss': 0.0193, 'learning_rate': 2.3566878980891724e-06, 'epoch': 4.78}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.6758822202682495, 'eval_accuracy': 0.853, 'eval_runtime': 7.24, 'eval_samples_per_second': 276.242, 'eval_steps_per_second': 8.702, 'epoch': 5.0}
{'train_runtime': 312.8985, 'train_samples_per_second': 79.898, 'train_steps_per_second': 2.509, 'train_loss': 0.17066853699410797, 'epoch': 5.0}


  0%|          | 0/63 [00:00<?, ?it/s]

## Cell 9: Fine-Tuning Task 2: Question Answering (SQuAD)

In [8]:
class BertForQuestionAnswering(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = TinyBERT(config)
        self.qa_outputs = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, start_positions=None, end_positions=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = self.qa_outputs(outputs)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)

        loss = None
        if start_positions is not None and end_positions is not None:
            ignored_index = start_logits.size(1)
            start_positions.clamp_(0, ignored_index)
            end_positions.clamp_(0, ignored_index)
            loss_fct = nn.CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            loss = (start_loss + end_loss) / 2
            
        return start_logits, end_logits, loss

In [ ]:
def postprocess_qa_predictions(examples, features, all_start_logits, all_end_logits, n_best_size=20, max_answer_length=30):
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = collections.OrderedDict()

    print("Post-processing ", len(examples), " examples split into ", len(features), " features.")

    for example_index, example in enumerate(tqdm(examples)):
        feature_indices = features_per_example[example_index]

        min_null_score = None
        valid_answers = []
        
        context = example["context"]
        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]
            cls_index = features[feature_index]["input_ids"].tolist().index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or min_null_score < feature_null_score:
                min_null_score = feature_null_score
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": context[start_char: end_char]
                        }
                    )
        
        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            best_answer = {"text": "", "score": 0.0}
        
        predictions[example["id"]] = best_answer["text"]

    return predictions

In [ ]:
squad_dataset = load_dataset('squad')
train_dataset = squad_dataset['train'].shuffle(seed=42).select(range(5000))
validation_dataset = squad_dataset['validation'].shuffle(seed=42).select(range(2000))

def preprocess_squad(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=config.max_position_embeddings,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )
    
    offset_mapping = inputs["offset_mapping"]
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i in range(len(examples['question'])):
        offset = offset_mapping[i]
        answer = answers[i]
        sequence_ids = inputs.sequence_ids(i)

        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
        
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            token_start_index = context_start
            while token_start_index <= context_end and offset[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            token_end_index = context_end
            while token_end_index >= context_start and offset[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)
            
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_train_dataset = train_dataset.map(preprocess_squad, batched=True, remove_columns=train_dataset.column_names)
tokenized_validation_dataset = validation_dataset.map(preprocess_squad, batched=True, remove_columns=validation_dataset.column_names)

tokenized_train_dataset.set_format('torch')
tokenized_validation_dataset.set_format('torch')
train_dataloader = DataLoader(tokenized_train_dataset, shuffle=True, batch_size=16)
validation_dataloader = DataLoader(tokenized_validation_dataset, batch_size=16)
qa_model = BertForQuestionAnswering(config).to(device)
qa_model.bert.load_state_dict(torch.load("tinybert_pretrained.pth"))

optimizer = AdamW(qa_model.parameters(), lr=3e-5)
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
scaler = torch.cuda.amp.GradScaler()

print("\nStarting Question Answering Fine-tuning...")
qa_model.train()
for epoch in range(num_epochs):
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        try:
            batch.pop("offset_mapping")
        except KeyError:
            pass

        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            _, _, loss = qa_model(**batch)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

print("Training complete.")
torch.save(qa_model.state_dict(), "tinybert_qa.pth")

print("\nStarting evaluation on the validation set...")
squad_metric = load("./squadf")
qa_model.eval()
eval_features = validation_dataset.map(
    preprocess_squad,
    batched=True,
    remove_columns=validation_dataset.column_names,
)
eval_features = eval_features.add_column("example_id", validation_dataset["id"])
eval_features.set_format(type='torch')

all_start_logits = []
all_end_logits = []
columns_to_remove = ["example_id", "offset_mapping", "start_positions", "end_positions"]
eval_dataloader = DataLoader(
    eval_features.remove_columns(columns_to_remove), 
    batch_size=16
)

for batch in tqdm(eval_dataloader, desc="Evaluating"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        start_logits, end_logits, _ = qa_model(**batch)
        all_start_logits.append(start_logits.cpu().numpy())
        all_end_logits.append(end_logits.cpu().numpy())

all_start_logits = np.concatenate(all_start_logits, axis=0)
all_end_logits = np.concatenate(all_end_logits, axis=0)
final_predictions = postprocess_qa_predictions(validation_dataset, eval_features, all_start_logits, all_end_logits)

formatted_predictions = [{"id": k, "prediction_text": v} for k, v in final_predictions.items()]
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in validation_dataset]

results = squad_metric.compute(predictions=formatted_predictions, references=references)

print("\n--- Evaluation Results ---")
print(f"Exact Match (EM): {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")


Starting Question Answering Fine-tuning...


Epoch 1:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/313 [00:00<?, ?it/s]

...

Training complete.

Starting evaluation on the validation set...


Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

Post-processing  2000  examples split into  2000  features.


  0%|          | 0/2000 [00:00<?, ?it/s]


--- Evaluation Results ---
Exact Match (EM): 0.50
F1 Score: 2.55


## Question Answering - Benchmark

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

squad_dataset = load_dataset('squad')
train_dataset = squad_dataset['train'].shuffle(seed=42).select(range(5000))
validation_dataset = squad_dataset['validation'].shuffle(seed=42).select(range(2000))
def preprocess_squad(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )
    offset_mapping = inputs["offset_mapping"]
    answers = examples["answers"]
    start_positions = []
    end_positions = []
    for i in range(len(examples['question'])):
        offset = offset_mapping[i]
        answer = answers[i]
        sequence_ids = inputs.sequence_ids(i)
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            token_start_index = context_start
            while token_start_index <= context_end and offset[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)
            token_end_index = context_end
            while token_end_index >= context_start and offset[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_train_dataset = train_dataset.map(preprocess_squad, batched=True, remove_columns=train_dataset.column_names)
tokenized_validation_dataset = validation_dataset.map(preprocess_squad, batched=True, remove_columns=validation_dataset.column_names)

tokenized_train_dataset.set_format('torch')
tokenized_validation_dataset.set_format('torch')

train_dataloader = DataLoader(tokenized_train_dataset, shuffle=True, batch_size=16)
validation_dataloader = DataLoader(tokenized_validation_dataset, batch_size=16)

benchmark_model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint).to(device)

optimizer = AdamW(benchmark_model.parameters(), lr=3e-5)
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
scaler = torch.cuda.amp.GradScaler()

print("\nStarting Benchmark Question Answering Fine-tuning...")
benchmark_model.train()
for epoch in range(num_epochs):
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        try:
            batch.pop("offset_mapping")
        except KeyError:
            pass
        
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            outputs = benchmark_model(**batch)
            loss = outputs.loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

print("Benchmark training complete.")
print("\nStarting evaluation on the benchmark model...")
squad_metric = load("./squadf")
benchmark_model.eval()

eval_features = validation_dataset.map(
    preprocess_squad,
    batched=True,
    remove_columns=validation_dataset.column_names,
)
eval_features = eval_features.add_column("example_id", validation_dataset["id"])
eval_features.set_format(type='torch')

all_start_logits = []
all_end_logits = []

columns_to_remove = ["example_id", "offset_mapping", "start_positions", "end_positions"]
eval_dataloader = DataLoader(
    eval_features.remove_columns(columns_to_remove), 
    batch_size=16
)

for batch in tqdm(eval_dataloader, desc="Evaluating"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = benchmark_model(**batch)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits
        all_start_logits.append(start_logits.cpu().numpy())
        all_end_logits.append(end_logits.cpu().numpy())

all_start_logits = np.concatenate(all_start_logits, axis=0)
all_end_logits = np.concatenate(all_end_logits, axis=0)

final_predictions = postprocess_qa_predictions(validation_dataset, eval_features, all_start_logits, all_end_logits)

formatted_predictions = [{"id": k, "prediction_text": v} for k, v in final_predictions.items()]
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in validation_dataset]

results = squad_metric.compute(predictions=formatted_predictions, references=references)

print("\n--- BENCHMARK Evaluation Results ---")
print(f"Benchmark Model: {model_checkpoint}")
print(f"Exact Match (EM): {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

/home/cseru/.pyenv/versions/3.10.12/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting Benchmark Question Answering Fine-tuning...


Epoch 1:   0%|          | 0/313 [00:00<?, ?it/s]

...

Benchmark training complete.

Starting evaluation on the benchmark model...


Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

Post-processing  2000  examples split into  2000  features.


  0%|          | 0/2000 [00:00<?, ?it/s]


--- BENCHMARK Evaluation Results ---
Benchmark Model: distilbert-base-uncased
Exact Match (EM): 57.90
F1 Score: 69.02


## Cell 10: Fine-Tuning Task 3: Semantic Similarity (SNLI)

In [37]:
snli_dataset = load_dataset('snli')
train_dataset = snli_dataset['train'].filter(lambda ex: ex['label'] != -1).shuffle(seed=42).select(range(5000))
test_dataset = snli_dataset['test'].filter(lambda ex: ex['label'] != -1).shuffle(seed=42).select(range(2000))

def tokenize_snli(examples):
    return tokenizer(examples['premise'], examples['hypothesis'], truncation=True, max_length=config.max_position_embeddings, padding='max_length')

train_dataset = train_dataset.map(tokenize_snli, batched=True, remove_columns=['premise', 'hypothesis'])
test_dataset = test_dataset.map(tokenize_snli, batched=True, remove_columns=['premise', 'hypothesis'])
train_dataset.set_format('torch')
test_dataset.set_format('torch')

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32)
test_dataloader = DataLoader(test_dataset, batch_size=32)

snli_model = BertForSequenceClassification(config, num_labels=3).to(device)
snli_model.bert.load_state_dict(torch.load("tinybert_pretrained.pth"))

optimizer = AdamW(snli_model.parameters(), lr=2e-5)
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
scaler = torch.cuda.amp.GradScaler()

print("\nStarting Semantic Similarity Fine-tuning...")
snli_model.train()
for epoch in range(num_epochs):
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            _, loss = snli_model(**batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

snli_model.eval()
total_correct = 0
total_samples = 0
for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        logits, _ = snli_model(**batch)
    predictions = torch.argmax(logits, dim=-1)
    total_correct += (predictions == batch['label']).sum().item()
    total_samples += len(batch['label'])

tinybert_snli_accuracy = total_correct / total_samples
print(f"TinyBERT Semantic Similarity Accuracy: {tinybert_snli_accuracy:.4f}")
torch.save(snli_model.state_dict(), "tinybert_snli.pth")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Starting Semantic Similarity Fine-tuning...


Epoch 1:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/157 [00:00<?, ?it/s]

TinyBERT Semantic Similarity Accuracy: 0.4485


## Cell 13: Benchmark - Semantic Similarity

In [ ]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments
import evaluate


model_checkpoint = "distilbert-base-uncased"
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

model = DistilBertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

training_args = TrainingArguments(
    output_dir="./results_snli",
    evaluation_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting Benchmark Semantic Similarity...")
trainer.train()
distilbert_snli_results = trainer.evaluate()

Using the latest cached version of the module from /home/cseru/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--accuracy/f887c0aab52c2d38e1f8a215681126379eca617f96c447638f751434e8e65b14 (last modified on Fri Aug 22 20:26:38 2025) since it couldn't be found locally at evaluate-metric--accuracy, or remotely on the Hugging Face Hub.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'pre_classifier.bias', 'pre_classifier.weight', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting Benchmark Semantic Similarity...


  0%|          | 0/625 [00:00<?, ?it/s]

{'loss': 1.0726, 'learning_rate': 4.600000000000001e-05, 'epoch': 0.4}
{'loss': 0.8335, 'learning_rate': 4.2080000000000004e-05, 'epoch': 0.8}


  0%|          | 0/32 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

{'eval_loss': 0.8422676920890808, 'eval_accuracy': 0.737, 'eval_runtime': 3.4961, 'eval_samples_per_second': 286.032, 'eval_steps_per_second': 9.153, 'epoch': 4.0}
{'loss': 0.1039, 'learning_rate': 6.08e-06, 'epoch': 4.4}
{'loss': 0.123, 'learning_rate': 2.08e-06, 'epoch': 4.8}


  0%|          | 0/32 [00:00<?, ?it/s]

{'eval_loss': 0.9024494886398315, 'eval_accuracy': 0.74, 'eval_runtime': 3.4973, 'eval_samples_per_second': 285.935, 'eval_steps_per_second': 9.15, 'epoch': 5.0}
{'train_runtime': 240.0751, 'train_samples_per_second': 83.307, 'train_steps_per_second': 2.603, 'train_loss': 0.426987622833252, 'epoch': 5.0}


  0%|          | 0/32 [00:00<?, ?it/s]

## Inference for Sentiment Classification

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sentiment_model = BertForSequenceClassification(config, num_labels=2).to(device)
sentiment_model.load_state_dict(torch.load("tinybert_sentiment.pth"))
sentiment_model.eval()

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config.max_position_embeddings,
        padding="max_length"
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits, _ = sentiment_model(**inputs)
    
    probabilities = F.softmax(logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    labels = ["Negative", "Positive"]
    confidence = probabilities[0][prediction].item()
    
    print(f"Input Text: '{text}'")
    print(f"Predicted Sentiment: {labels[prediction]}")
    print(f"Confidence: {confidence:.4f}")

predict_sentiment("Finally, we’re completing our graduation — it feels amazing!")
print("-" * 30)
predict_sentiment("But graduation took fucking six years.")

Input Text: 'Finally, we’re completing our graduation — it feels amazing!'
Predicted Sentiment: Positive
Confidence: 0.6377
------------------------------
Input Text: 'But graduation took fucking six years.'
Predicted Sentiment: Negative
Confidence: 0.8021


## Inference for QA

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

qa_model = BertForQuestionAnswering(config).to(device)
qa_model.load_state_dict(torch.load("tinybert_qa.pth"))
qa_model.eval()

def answer_question(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=config.max_position_embeddings,
        padding="max_length",
        return_offsets_mapping=True
    )
    
    offset_mapping = inputs.pop("offset_mapping")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        start_logits, end_logits, _ = qa_model(**inputs)
    
    start_index = torch.argmax(start_logits, dim=1).item()
    end_index = torch.argmax(end_logits, dim=1).item()
    start_char, _ = offset_mapping[0][start_index]
    _, end_char = offset_mapping[0][end_index]
    
    answer_text = context[start_char:end_char]
    
    print(f"Context: '{context}'")
    print(f"Question: '{question}'")
    print(f"Predicted Answer: '{answer_text}'")

context = "Pirates of the Caribbean features several iconic characters. The main character is Captain Jack Sparrow"
question = "Who is the main character?"

answer_question(question, context)
print("-" * 30)

context = "Elizabeth Swann is a brave and intelligent woman from Port Royal who plays a central role in the Pirates of the Caribbean"
question = "Where does Elizabeth come from?"
answer_question(question, context)

Context: 'Pirates of the Caribbean features several iconic characters. The main character is Captain Jack Sparrow'
Question: 'Who is the main character?'
Predicted Answer: 'The main character is Captain'
------------------------------
Context: 'Elizabeth Swann is a brave and intelligent woman from Port Royal who plays a central role in the Pirates of the Caribbean'
Question: 'Where does Elizabeth come from?'
Predicted Answer: 'Port Royal who plays a central role in the Pirates of the Caribbean'


## Inference for semantic similarity

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
snli_model = BertForSequenceClassification(config, num_labels=3).to(device)
snli_model.load_state_dict(torch.load("tinybert_snli.pth"))
snli_model.eval()
def predict_relationship(premise, hypothesis):
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=config.max_position_embeddings,
        padding="max_length"
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits, _ = snli_model(**inputs)
    
    probabilities = F.softmax(logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    labels = ["Entailment", "Neutral", "Contradiction"]
    confidence = probabilities[0][prediction].item()
    
    print(f"Premise: '{premise}'")
    print(f"Hypothesis: '{hypothesis}'")
    print(f"Predicted Relationship: {labels[prediction]}")
    print(f"Confidence: {confidence:.4f}")

predict_relationship(
    "A man is playing a guitar on a stage.",
    "A man is sleeping in a bed."
)
print("-" * 30)
predict_relationship(
    "A dog is chasing a cat through a garden.",
    "An animal is chasing another animal."
)
print("-" * 30)
predict_relationship(
    "The woman is wearing a blue dress.",
    "She has blonde hair."
)

Premise: 'A man is playing a guitar on a stage.'
Hypothesis: 'A man is sleeping in a bed.'
Predicted Relationship: Contradiction
Confidence: 0.5629
------------------------------
Premise: 'A dog is chasing a cat through a garden.'
Hypothesis: 'An animal is chasing another animal.'
Predicted Relationship: Contradiction
Confidence: 0.4352
------------------------------
Premise: 'The woman is wearing a blue dress.'
Hypothesis: 'She has blonde hair.'
Predicted Relationship: Entailment
Confidence: 0.4875
